# FraudGuard HK - RAG System Setup

This notebook sets up the RAG (Retrieval-Augmented Generation) database for fraud detection.

## Prerequisites
- Python 3.11+ (Python 3.14 may have compatibility warnings but should work)
- Virtual environment activated
- Dependencies installed: `pip install -r data-collection/requirements.txt`

## What this notebook does
1. Scrapes HK01 for fraud/scam news articles
2. Creates vector embeddings for each article
3. Stores them in a Chroma database for RAG retrieval

## Step 1: Import Required Modules

In [ ]:
import os
import sys

# Import rag_core from the project root
from rag_core import RAGDatabase, FraudAnalyzer, create_rag_system

print("Core modules imported successfully!")

## Step 2: Check if Database Already Exists

If you already have the Chroma database (`chroma_hk01_scam_db` folder), you can skip to Step 5.

In [ ]:
db_path = "./chroma_hk01_scam_db"

if os.path.exists(db_path):
    print(f"[OK] Database already exists at: {db_path}")
    print("\n>>> You can skip to Step 5 to test the existing database. <<<")
else:
    print("[MISSING] Database not found.")
    print("\n>>> Please run Steps 3-4 to create the database. <<<")

## Step 3: Scrape HK01 for Fraud Articles (Only if database doesn't exist)

This will fetch fraud-related articles from HK01 and save them to `data/hk01_scam_articles.md` and `data/hk01_scam_articles.json`.

In [ ]:
# Run the scraper directly using the helper function
import importlib.util

# Load the scraper module from the hyphenated folder
scraper_path = os.path.join('data-collection', 'hk01_scraper.py')
spec = importlib.util.spec_from_file_location("hk01_scraper", scraper_path)
hk01_scraper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hk01_scraper)

# Run the scraper (this fetches articles and saves to data/ folder)
print("Starting HK01 scraper...")
print("This may take several minutes...\n")

# Fetch articles (default: 1000 articles)
articles = hk01_scraper.run_scraper(max_articles=1000, output_format="both")

print(f"\nCompleted! Fetched {len(articles)} articles.")

## Step 4: Build the Chroma Vector Database

This creates embeddings for all articles and stores them in Chroma.

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from sentence_transformers import SentenceTransformer
import json

# Load articles from JSON
print("Loading articles from JSON...")
with open("data/hk01_scam_articles.json", "r", encoding="utf-8") as f:
    data = json.load(f)
    articles = data.get('articles', [])

print(f"Loaded {len(articles)} articles")

In [ ]:
# Create documents for Chroma
documents = []
for article in articles:
    content = article.get("content", "")
    title = article.get("title", "")
    
    # Combine title and content for better retrieval
    full_content = f"{title}\n\n{content}"
    
    doc = Document(
        page_content=full_content,
        metadata={
            "title": title,
            "url": article.get("url", ""),
            "date": article.get("publish_time", ""),
            "category": article.get("category", "")
        }
    )
    documents.append(doc)

print(f"Created {len(documents)} documents")

In [ ]:
# Initialize embedding model
class LocalEmbedding:
    def __init__(self, model_name="BAAI/bge-large-zh-v1.5"):
        print(f"Loading embedding model: {model_name}")
        print("This may take a few minutes on first run (downloading model)...\n")
        self.model = SentenceTransformer(model_name)
        print("Model loaded successfully!")
    
    def embed_documents(self, texts):
        return self.model.encode(texts, normalize_embeddings=True).tolist()
    
    def embed_query(self, text):
        return self.model.encode([text], normalize_embeddings=True)[0].tolist()

embedding = LocalEmbedding()

In [ ]:
# Create Chroma database
print("Creating Chroma database...")
print("This may take several minutes depending on your hardware.\n")

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding,
    persist_directory=db_path,
    collection_metadata={"hnsw:space": "cosine"}
)

print(f"\n{'='*50}")
print(f"Database created successfully!")
print(f"Total documents: {vectorstore._collection.count()}")
print(f"Location: {db_path}")
print(f"{'='*50}")

## Step 5: Test the RAG System

In [ ]:
# Load the database
print("Loading RAG database...")
rag_db, analyzer = create_rag_system(db_path)

print(f"Database loaded: {rag_db.get_case_count()} documents")

In [ ]:
# Test with a sample scam message
test_message = """
恭喜您！您已被選中獲得 HK$50,000 現金大獎！
請立即點擊以下連結領取：http://fake-link.com/prize
請在24小時內完成領取，否則獎品將作廢。
"""

print("Test message:")
print(test_message)

# Retrieve similar cases
similar_cases = rag_db.retrieve_similar_cases(test_message)

print(f"\nFound {len(similar_cases)} similar cases:")
for i, doc in enumerate(similar_cases, 1):
    print(f"{i}. {doc.metadata.get('title', 'N/A')[:50]}...")

# Analyze
result = analyzer.analyze(
    message=test_message,
    rag_cases=similar_cases,
    language="zh",
    elderly_mode=True
)

print("\n" + "="*50)
print("Analysis Result (Elderly Mode):")
print("="*50)
print(f"Risk Level: {result['visual_label']}")
print(f"Reason: {result['simple_reason']}")
print(f"Action: {result['action']}")
print(f"Confidence: {result['confidence_display']}")

## Step 6: Verify Database Files

Make sure these files/folders exist before running the web app:
- `chroma_hk01_scam_db/` - Chroma database folder (REQUIRED)
- `data/hk01_scam_articles.md` - Raw articles (optional)
- `data/hk01_scam_articles.json` - Articles in JSON format (optional)

In [ ]:
# Verify files
print("Checking required files...")
print()

all_good = True

if os.path.exists(db_path):
    print(f"[OK] Chroma database: {db_path}")
else:
    print(f"[MISSING] Chroma database: {db_path}")
    all_good = False

if os.path.exists("data/hk01_scam_articles.md"):
    print("[OK] Articles markdown: data/hk01_scam_articles.md")
else:
    print("[OPTIONAL] Articles markdown: data/hk01_scam_articles.md (not required)")

if os.path.exists("data/hk01_scam_articles.json"):
    print("[OK] Articles JSON: data/hk01_scam_articles.json")
else:
    print("[OPTIONAL] Articles JSON: data/hk01_scam_articles.json (not required)")

print()
if all_good:
    print("="*50)
    print("SUCCESS! Setup complete!")
    print("You can now run the web application:")
    print("  bun run dev")
    print("="*50)
else:
    print("[ERROR] Please run Steps 3-4 to create the database.")